In [1]:
import sys, os, glob as _g, subprocess
try:
    import onnxruntime; print(f'ort {onnxruntime.__version__} ready')
except ImportError:
    wdirs = {os.path.dirname(w) for w in _g.glob('/kaggle/input/**/*.whl', recursive=True)
             if 'onnxruntime' in os.path.basename(w)}
    if not wdirs: raise RuntimeError('no onnxruntime wheel in /kaggle/input')
    fl = [x for d in wdirs for x in ['--find-links', d]]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', *fl, 'onnxruntime'], check=True)
    import onnxruntime; print(f'ort {onnxruntime.__version__} installed')


ort 1.24.4 installed


# BirdCLEF 2026 Inference v27 — Soundscape Fine-tuned GRU

**Score history**
| Version | LB |
|---------|----|
| v23 GRU-only | **0.858** |
| v26 gru=0.8/mel=0.2 | 0.857 |
| mel-only | 0.741 |

Mel branch adds no LB value, so v27 uses **GRU-only** (`gru_weight=1.0`).
v27 GRU checkpoints are fine-tuned from v23 on soundscape-only data (LR=2e-5, 10 epochs).

**Required datasets**
1. `birdclef-2026` (competition)
2. `chiragggg/birdclef-2026-perch-onnx`
3. `chiragggg/birdclef-2026-perch-weights-v27`


In [ ]:
import os, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa, onnxruntime as ort
from scipy.ndimage import gaussian_filter1d
import torch, torch.nn as nn
from torch.cuda.amp import autocast
from tqdm import tqdm
warnings.filterwarnings('ignore')

CFG = dict(
    folds=5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    # Perch embedding params (must match ONNX model)
    perch_sr=32000, perch_seconds=5, perch_emb_dim=1536, perch_batch=16,
    # GRU architecture — must match v27/v23 training exactly
    gru_hidden=512,
    gru_layers=2,
    # v27: GRU-only (mel adds no LB value)
    gru_weight=1.0,
    gauss_sigma=1.0,
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']  # 160000
device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device: {device}  ort: {ort.__version__}')
print(f'GRU v27  hidden={CFG["gru_hidden"]}  layers={CFG["gru_layers"]}')
print(f'gru_weight={CFG["gru_weight"]}  (mel branch disabled)')


In [ ]:
def _fe(*c):
    return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                   '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
TEST_AUDIO   = _fe('/kaggle/input/birdclef-2026/test_soundscapes',
                   '/kaggle/input/competitions/birdclef-2026/test_soundscapes')
SAMPLE_SUB   = _fe('/kaggle/input/birdclef-2026/sample_submission.csv',
                   '/kaggle/input/competitions/birdclef-2026/sample_submission.csv')

# v27 soundscape-fine-tuned GRU checkpoints
GRU_CKPT_DIR = _fe('/kaggle/input/birdclef-2026-perch-weights-v27',
                   '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v27',
                   '/kaggle/working')

ONNX_PATH = None
for _c in [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx',
]:
    if os.path.exists(_c):
        ONNX_PATH = _c
        break

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
n_classes   = len(species)
sp_idx      = {l: i for i, l in enumerate(species)}
print(f'Species      : {n_classes}')
print(f'GRU_CKPT_DIR : {GRU_CKPT_DIR}')
print(f'ONNX_PATH    : {ONNX_PATH}')


In [ ]:
class PerchGRU(nn.Module):
    """v27/v23 architecture: hidden=512, layers=2, bidirectional."""
    def __init__(self, n_classes, emb_dim=1536, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(512, hidden, n_layers, batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )

    def forward(self, x):
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out


print('PerchGRU defined')


In [ ]:
def _load_gru(names, ckpt_dir):
    ms = []
    for n in names:
        p = Path(ckpt_dir) / n
        if not p.exists():
            print(f'  MISSING: {p}')
            continue
        m = PerchGRU(n_classes, emb_dim=CFG['perch_emb_dim'],
                     hidden=CFG['gru_hidden'], n_layers=CFG['gru_layers']).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True))
        m.eval()
        ms.append(m)
        print(f'  OK {n}')
    return ms

print('Loading v27 GRU checkpoints (hidden=512, layers=2)...')
gru_models = _load_gru(
    [f'perch_gru_v27_fold{i}.pt' for i in range(CFG['folds'])],
    GRU_CKPT_DIR,
)
print(f'Loaded GRU: {len(gru_models)}/5')
if len(gru_models) == 0:
    print('WARNING: no v27 checkpoints found -- check GRU_CKPT_DIR and checkpoint names.')


In [6]:
_sess = None; _inp = None; _eidx = 0; _onnx_ok = False
if ONNX_PATH is None:
    print('ONNX not found -- GRU branch will be disabled')
else:
    try:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = os.cpu_count() or 4
        _sess = ort.InferenceSession(ONNX_PATH, sess_options=opts,
                                     providers=['CPUExecutionProvider'])
        _inp  = _sess.get_inputs()[0].name
        _out_names = [o.name for o in _sess.get_outputs()]
        ekey  = next((o.name for o in _sess.get_outputs()
                      if o.shape and o.shape[-1] == 1536), _out_names[0])
        _eidx = _out_names.index(ekey)
        _t    = _sess.run(None, {_inp: np.zeros((1, CFG['perch_target']), np.float32)})
        _e    = _t[_eidx]
        if _e.ndim == 3:
            _e = _e.mean(1)
        assert _e.shape[-1] == 1536, f'Expected 1536-d emb, got {_e.shape}'
        _onnx_ok = True
        print(f'ONNX OK: {Path(ONNX_PATH).name}  ekey={ekey}  emb={_e.shape}')
    except Exception as ex:
        print(f'ONNX ERROR: {ex}')


ONNX OK: perch_v2.onnx  ekey=embedding  emb=(1, 1536)


In [ ]:
_amp = (device.type == 'cuda')


def _embs(path, ends):
    """Run Perch ONNX on audio -> {end_sec: emb_1536} dict."""
    if not _onnx_ok or not ends:
        return {}
    try:
        y, sr = sf.read(path, always_2d=False)
        if y.ndim == 2:
            y = y.mean(1)
        if sr != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
    except Exception as e:
        print(f'[W] perch read: {e}')
        return {}
    clips = []
    for es in ends:
        e0 = int(es * CFG['perch_sr'])
        s0 = max(0, e0 - CFG['perch_target'])
        c  = y[s0:e0]
        if len(c) < CFG['perch_target']:
            c = np.pad(c, (0, CFG['perch_target'] - len(c)))
        clips.append(c)
    all_embs = []
    for bi in range(0, len(clips), CFG['perch_batch']):
        B   = np.stack(clips[bi:bi + CFG['perch_batch']])
        out = _sess.run(None, {_inp: B})[_eidx]
        if out.ndim == 3:
            out = out.mean(1)
        all_embs.append(out.astype(np.float32))
    return dict(zip(ends, np.vstack(all_embs)))


def predict_v27(path, ends):
    """GRU-only prediction. Returns (T, n_classes) probability array."""
    T = len(ends)
    if not gru_models or not _onnx_ok:
        return np.full((T, n_classes), 0.5, np.float32)

    em  = _embs(path, ends)
    seq = torch.from_numpy(
        np.stack([em.get(e, np.zeros(CFG['perch_emb_dim'], np.float32)) for e in ends])
    ).float().unsqueeze(0).to(device)  # (1, T, 1536)

    preds = []
    for m in gru_models:
        with torch.inference_mode(), autocast(enabled=_amp):
            preds.append(torch.sigmoid(m(seq).float())[0].cpu().numpy())

    p = np.mean(preds, axis=0)  # (T, n_classes) — fold average

    if T > 1 and CFG['gauss_sigma'] > 0:
        p = gaussian_filter1d(p.astype(np.float64), sigma=CFG['gauss_sigma'], axis=0).astype(np.float32)
    return p


print(f'predict_v27 defined  GRU folds={len(gru_models)}  onnx_ok={_onnx_ok}')


In [ ]:
sub = pd.read_csv(SAMPLE_SUB).copy()
sub['_sc'] = sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Rows: {len(sub)}')

row_ids = []; probs_list = []; n_miss = 0; n_err = 0

for sc, grp in tqdm(sub.groupby('_sc'), desc='soundscapes', unit='f'):
    rids = [str(r) for r in grp['row_id']]
    ap = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TEST_AUDIO) / f'{sc}{ext}'
        if c.exists():
            ap = str(c)
            break
    if ap is None:
        n_miss += 1
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))
        continue
    try:
        ends = [int(r.rsplit('_', 1)[-1]) for r in rids]
    except Exception:
        n_err += 1
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))
        continue
    try:
        p = predict_v27(ap, ends)
        row_ids.extend(rids)
        probs_list.append(p)
    except Exception as e:
        n_err += 1
        print(f'ERR {sc}: {e}')
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))

print(f'Done  missing={n_miss}  errors={n_err}')


In [ ]:
mat = np.concatenate(probs_list, axis=0)
mu, sd = mat.mean(), mat.std()
if abs(mu - 0.5) < 0.001 and sd < 0.01:
    print(f'WARNING: all-neutral predictions (mean={mu:.4f}, std={sd:.4f})')
    print('  Check Cell 6 (ONNX) and Cell 5 (checkpoints).')
else:
    print(f'OK  mean={mu:.4f}  std={sd:.4f}')

sub_df = pd.DataFrame(mat, columns=species)
sub_df.insert(0, 'row_id', row_ids)
cols   = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[cols]
sub_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved  shape={sub_df.shape}')
print(f'v27 GRU-only  gru_weight={CFG["gru_weight"]}  '
      f'tta={CFG["mel_tta"]}  '
      f'folds={len(gru_models)}  gauss_sigma={CFG["gauss_sigma"]}')
sub_df.head(3)
